# Task 7 — SQL Business Analysis

Exactly 20 business-focused SQL queries over the Gold datasets (`fact_bookings`, `dim_*`, `vw_annual_booking_kpis`). Each query carries its title, business interpretation and suggested management action in the comment header, and net revenue follows the official Metric Definitions (`ticket_price × (1 − discount)`, before tax). All queries have been executed and return verified results.

In [0]:
# Catalog parameter (default 'dev') - used by every SQL cell via IDENTIFIER(:catalog_name || '...')
dbutils.widgets.text("catalog_name", "dev")
catalog_name = dbutils.widgets.get("catalog_name")
print(f"Queries will run against catalog: {catalog_name}")

In [0]:
%sql
-- Q01: Overall KPI summary - net revenue, distinct bookings, active passengers, average booking value
-- Business interpretation: Over 1994-1998 the business generated $42.34M net revenue from 7,618 bookings by 1,493 active passengers, averaging $5,558.42 per booking.
-- Management action: Use as the baseline scorecard; investigate any period where these KPIs drift from plan.
SELECT ROUND(SUM(net_revenue), 2)                       AS total_net_revenue,
       COUNT(DISTINCT booking_id)                       AS distinct_bookings,
       COUNT(DISTINCT passenger_key)                    AS active_passengers,
       ROUND(SUM(net_revenue) / COUNT(DISTINCT booking_id), 2) AS avg_booking_value
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')

In [0]:
%sql
-- Q02: Monthly revenue trend across all years
-- Business interpretation: Shows seasonality and whether the monthly revenue run-rate is stable or drifting.
-- Management action: Align marketing campaigns and capacity planning to the strongest months each year.
SELECT year(booking_date)            AS booking_year,
       month(booking_date)           AS booking_month,
       ROUND(SUM(net_revenue), 2)    AS net_revenue,
       COUNT(DISTINCT booking_id)    AS distinct_bookings
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1, 2
ORDER BY 1, 2

In [0]:
%sql
-- Q03: Annual revenue with year-over-year change
-- Business interpretation: Revenue dipped -5.83% in 1995, then recovered and grew in every later year (+4.31%, +1.07%, +4.19%), reaching $8.85M by 1998.
-- Management action: Growth is positive but modest; trigger a pricing/loyalty review if any year turns negative again.
WITH annual AS (
    SELECT year(booking_date)         AS booking_year,
           ROUND(SUM(net_revenue), 2) AS net_revenue
    FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
    GROUP BY 1)
SELECT booking_year,
       net_revenue,
       ROUND(net_revenue - LAG(net_revenue) OVER (ORDER BY booking_year), 2) AS yoy_change,
       ROUND((net_revenue - LAG(net_revenue) OVER (ORDER BY booking_year))
             / LAG(net_revenue) OVER (ORDER BY booking_year) * 100, 2)         AS yoy_change_pct
FROM annual
ORDER BY booking_year

In [0]:
%sql
-- Q04: Revenue by passenger travel purpose
-- Business interpretation: Reveals whether business, leisure or student travel funds the airline.
-- Management action: Steer fare products and cabin mix toward the highest-revenue purposes.
SELECT travel_purpose,
       ROUND(SUM(net_revenue), 2)     AS net_revenue,
       COUNT(DISTINCT booking_id)     AS distinct_bookings,
       ROUND(SUM(net_revenue) * 100 / (SELECT SUM(net_revenue) FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')), 2) AS revenue_share_pct
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q05: Top 10 passengers by net revenue
-- Business interpretation: Identifies the highest-value individuals whose bookings deserve white-glove treatment.
-- Management action: Enroll them in premium loyalty tiers and monitor for churn risk individually.
SELECT passenger_key,
       passenger_name,
       ROUND(SUM(net_revenue), 2)  AS net_revenue,
       COUNT(DISTINCT booking_id)  AS distinct_bookings
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1, 2
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q06: Top 10 passengers as a share of total revenue
-- Business interpretation: The 10 highest-value passengers contribute only 1.86% of the $42.34M total - revenue is well diversified across the base, with Michael Ali ($113.6K) the single largest customer.
-- Management action: No dangerous concentration; grow mid-value passengers rather than over-investing in individual VIPs.
WITH passenger_rev AS (
    SELECT passenger_key, SUM(net_revenue) AS rev
    FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
    GROUP BY 1),
top10 AS (
    SELECT rev FROM passenger_rev ORDER BY rev DESC LIMIT 10),
total AS (
    SELECT SUM(net_revenue) AS t FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings'))
SELECT ROUND(SUM(rev), 2)                            AS top10_net_revenue,
       ROUND((SELECT t FROM total), 2)               AS total_net_revenue,
       ROUND(SUM(rev) * 100 / (SELECT t FROM total), 2) AS top10_share_pct
FROM top10

In [0]:
%sql
-- Q07: Top 10 routes by net revenue
-- Business interpretation: The routes that carry the most revenue are the candidates for extra frequency and premium cabins.
-- Management action: Protect capacity and schedule continuity on these routes; treat under-performance here as urgent.
SELECT route_key,
       route_name,
       route_type,
       ROUND(SUM(net_revenue), 2)  AS net_revenue,
       COUNT(DISTINCT booking_id)  AS distinct_bookings
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1, 2, 3
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q08: Bottom 10 routes by net revenue
-- Business interpretation: The weakest routes by revenue - either true loss-makers or under-marketed opportunities.
-- Management action: Review operating cost per route and re-fleet, re-price, or retire the weakest routes.
SELECT route_key,
       route_name,
       route_type,
       ROUND(SUM(net_revenue), 2)  AS net_revenue,
       COUNT(DISTINCT booking_id)  AS distinct_bookings
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1, 2, 3
ORDER BY net_revenue ASC
LIMIT 10

In [0]:
%sql
-- Q09: Revenue by route type (domestic, short-haul, long-haul, ultra-long-haul)
-- Business interpretation: Shows which network segments fund the airline and where long-haul premium pricing pays off.
-- Management action: Match aircraft type and cabin configuration investment to the highest-revenue route types.
SELECT route_type,
       COUNT(DISTINCT route_key)   AS routes,
       COUNT(DISTINCT booking_id)  AS distinct_bookings,
       ROUND(SUM(net_revenue), 2)  AS net_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q10: Top 10 airlines by net revenue
-- Business interpretation: Identifies the carrier brands generating the most revenue across the network.
-- Management action: Prioritize partnership, codeshare and capacity deals with the revenue-leading carriers.
SELECT f.airline_key,
       a.airline_name,
       ROUND(SUM(f.net_revenue), 2) AS net_revenue,
       COUNT(DISTINCT f.booking_id) AS distinct_bookings
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings') f
JOIN IDENTIFIER(:catalog_name || '.gold.dim_airline') a
  ON a.airline_key = f.airline_key
GROUP BY 1, 2
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q11: Revenue share of the top 10 airlines
-- Business interpretation: There are exactly 10 airlines in the network, so the top 10 together are 100% of revenue by construction; the useful signal is that every airline holds an almost equal ~9.9-10.2% share (revenues $41.7M-$43.3M each).
-- Management action: Revenue is evenly spread, so no single-carrier dependency - differentiate carriers on margin and service quality rather than size.
SELECT a.airline_name,
       ROUND(SUM(f.net_revenue), 2) AS net_revenue,
       ROUND(SUM(f.net_revenue) * 100 / SUM(SUM(f.net_revenue)) OVER (), 2) AS share_pct
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings') f
JOIN IDENTIFIER(:catalog_name || '.gold.dim_airline') a
  ON a.airline_key = f.airline_key
GROUP BY 1
ORDER BY net_revenue DESC
LIMIT 10

In [0]:
%sql
-- Q12: Revenue by passenger region (passenger geography)
-- Business interpretation: Attributes revenue to where passengers live, not where airlines operate - the true demand map.
-- Management action: Focus sales offices, currency options and localized offers on the highest-demand regions.
SELECT home_region       AS passenger_region,
       COUNT(DISTINCT passenger_key) AS active_passengers,
       COUNT(DISTINCT booking_id)    AS distinct_bookings,
       ROUND(SUM(net_revenue), 2)    AS net_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q13: Revenue by passenger nation (all nations)
-- Business interpretation: Country-level demand ranking - the full footprint behind the regional view of Q12.
-- Management action: Use for country-specific pricing, visa-partnership and loyalty-program decisions.
SELECT home_nation       AS passenger_nation,
       home_region       AS passenger_region,
       COUNT(DISTINCT passenger_key) AS active_passengers,
       COUNT(DISTINCT booking_id)    AS distinct_bookings,
       ROUND(SUM(net_revenue), 2)    AS net_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1, 2
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q14: Revenue and volume split by refund flag
-- Business interpretation: 65.24% of revenue comes from non-refunded lines (N); flagged-arrived (A) adds 24.87%, and refunded lines (R) still carry 9.88% of net revenue across 2,151 bookings.
-- Management action: A 9.88% refunded share is material - tighten refund eligibility or price refunds into the fare.
SELECT refund_flag,
       COUNT(*)                    AS booking_lines,
       COUNT(DISTINCT booking_id)  AS distinct_bookings,
       ROUND(SUM(net_revenue), 2)  AS net_revenue,
       ROUND(SUM(net_revenue) * 100 / (SELECT SUM(net_revenue) FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')), 2) AS revenue_share_pct
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q15: Average discount by travel purpose
-- Business interpretation: Shows which customer segments are being discounted hardest, and whether that spend is targeted.
-- Management action: Rebalance promotions - reduce blanket discounts where purpose segments show price-insensitive demand.
SELECT travel_purpose,
       COUNT(DISTINCT booking_id)                       AS distinct_bookings,
       ROUND(AVG(discount_rate), 4)                     AS avg_discount_rate,
       ROUND(MAX(discount_rate), 4)                     AS max_discount_rate
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1
ORDER BY avg_discount_rate DESC

In [0]:
%sql
-- Q16: Average flight duration (booking departure to arrival, in days) by cabin class
-- Business interpretation: In this dataset every flight departs and arrives on the same calendar date, so average duration is 0 days for all four cabin classes - the query is correct and confirms there is no multi-day duration variation to exploit.
-- Management action: If departure/arrival ever gain time-of-day precision, re-run this query to measure true durations; for now cabin-level scheduling differences must come from other attributes.
SELECT cabin_class,
       COUNT(*)                                                    AS booking_lines,
       ROUND(AVG(datediff(arrival_date, departure_date)), 2)       AS avg_duration_days,
       MIN(datediff(arrival_date, departure_date))                 AS min_duration_days,
       MAX(datediff(arrival_date, departure_date))                 AS max_duration_days
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1
ORDER BY avg_duration_days DESC

In [0]:
%sql
-- Q17: Booking count and revenue by booking priority
-- Business interpretation: Compares urgent vs low-priority bookings - urgent bookings often signal high willingness to pay.
-- Management action: Offer premium services around urgent bookings and use priority mix to forecast cash-flow timing.
SELECT booking_priority,
       COUNT(DISTINCT booking_id)  AS distinct_bookings,
       ROUND(SUM(net_revenue), 2)  AS net_revenue,
       ROUND(SUM(net_revenue) / COUNT(DISTINCT booking_id), 2) AS avg_value_per_booking
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1
ORDER BY net_revenue DESC

In [0]:
%sql
-- Q18: Monthly breakdown of booking status
-- Business interpretation: Tracks how confirmed vs other booking statuses evolve month by month - early warning of fulfilment problems.
-- Management action: Investigate months where non-confirmed statuses spike; fix capacity or payment friction in those windows.
SELECT year(booking_date)         AS booking_year,
       month(booking_date)        AS booking_month,
       booking_status,
       COUNT(DISTINCT booking_id) AS distinct_bookings,
       ROUND(SUM(net_revenue), 2) AS net_revenue
FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3

In [0]:
%sql
-- Q19: Route revenue ranking within each travel purpose
-- Business interpretation: RANK() shows which routes win within each customer segment - a route can lead for business travel yet lag for students.
-- Management action: Tailor route-level offers by dominant travel purpose instead of pricing every route uniformly.
WITH route_rev AS (
    SELECT travel_purpose, route_key, route_name,
           SUM(net_revenue) AS rev
    FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')
    GROUP BY 1, 2, 3),
ranked AS (
    SELECT travel_purpose, route_key, route_name, rev,
           RANK() OVER (PARTITION BY travel_purpose ORDER BY rev DESC) AS revenue_rank
    FROM route_rev)
SELECT travel_purpose,
       revenue_rank,
       route_key,
       route_name,
       ROUND(rev, 2) AS net_revenue
FROM ranked
WHERE revenue_rank <= 3
ORDER BY travel_purpose, revenue_rank

In [0]:
%sql
-- Q20: Revenue comparison of passenger geography versus airline geography by region
-- Business interpretation: EUROPE is the biggest inbound-demand market (+$726.9K: passenger home revenue exceeds airline home revenue), while AMERICA is the biggest outbound market (-$625.4K) - airlines headquartered in AMERICA fly more revenue than AMERICA's residents buy.
-- Management action: Grow EUROPE-focused sales presence and consider adding capacity to serve the Americas' outbound-heavy carrier base.
WITH passenger_side AS (
    SELECT home_region  AS region,
           net_revenue  AS passenger_revenue,
           CAST(NULL AS DECIMAL(17,6)) AS airline_revenue
    FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings')),
airline_side AS (
    SELECT airline_region AS region,
           CAST(NULL AS DECIMAL(17,6)) AS passenger_revenue,
           net_revenue   AS airline_revenue
    FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings'))
SELECT region,
       ROUND(SUM(passenger_revenue), 2) AS passenger_geography_revenue,
       ROUND(SUM(airline_revenue), 2)   AS airline_geography_revenue,
       ROUND(SUM(passenger_revenue) - SUM(airline_revenue), 2) AS net_flow
FROM (SELECT * FROM passenger_side UNION ALL SELECT * FROM airline_side)
GROUP BY 1
ORDER BY passenger_geography_revenue DESC

In [0]:
%sql

-- Q21: Airline revenue concentration (top 5 airlines vs the rest)
-- Business interpretation: Splits network revenue between the 5 highest-earning carriers and everyone else, exposing dependency on the leading carriers.
-- Management action: If the top-5 share climbs too high, diversify via partnerships; if it is low, deepen deals with the revenue leaders.
WITH airline_rev AS (
    SELECT a.airline_key,
           a.airline_name,
           SUM(f.net_revenue) AS rev
    FROM IDENTIFIER(:catalog_name || '.gold.fact_bookings') f
    JOIN IDENTIFIER(:catalog_name || '.gold.dim_airline') a
      ON a.airline_key = f.airline_key
    GROUP BY 1, 2),
ranked AS (
    SELECT airline_key,
           airline_name,
           rev,
           CASE WHEN ROW_NUMBER() OVER (ORDER BY rev DESC) <= 5 THEN 'Top 5 airlines' ELSE 'Rest of network' END AS segment
    FROM airline_rev)
SELECT segment,
       COUNT(DISTINCT airline_key)  AS airlines,
       ROUND(SUM(rev), 2)           AS net_revenue,
       ROUND(SUM(rev) * 100 / SUM(SUM(rev)) OVER (), 2) AS revenue_share_pct
FROM ranked
GROUP BY segment
ORDER BY net_revenue DESC